In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/notebooks/jigyasadoharey/scratch-model/__results__.html
/kaggle/input/notebooks/jigyasadoharey/scratch-model/word2idx.pkl
/kaggle/input/notebooks/jigyasadoharey/scratch-model/__notebook__.ipynb
/kaggle/input/notebooks/jigyasadoharey/scratch-model/__output__.json
/kaggle/input/notebooks/jigyasadoharey/scratch-model/model.pt
/kaggle/input/notebooks/jigyasadoharey/scratch-model/custom.css
/kaggle/input/notebooks/jigyasadoharey/scratch-model/repo/README.md
/kaggle/input/notebooks/jigyasadoharey/scratch-model/repo/DL-24f2001460-notebook-t22026.ipynb
/kaggle/input/notebooks/jigyasadoharey/scratch-model/repo/models/tfidf.py
/kaggle/input/notebooks/jigyasadoharey/scratch-model/repo/models/scratch.py
/kaggle/input/notebooks/jigyasadoharey/scratch-model/repo/models/__pycache__/scratch.cpython-312.pyc
/kaggle/input/notebooks/jigyasadoharey/scratch-model/repo/wandb/run-20260630_152526-f1dfl9xy/run-f1dfl9xy.wandb
/kaggle/input/notebooks/jigyasadoharey/scratch-model/repo/wandb/run-2026

In [2]:
import os
os.environ["wandb_key"] = "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"

In [3]:
import numpy as np
!pip install -q -U bitsandbytes>=0.46.1
import pandas as pd
import torch
import os
import wandb
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

try:
    from kaggle_secrets import UserSecretsClient
    wandb_key = UserSecretsClient().get_secret("wandb_key")
except Exception:
    wandb_key = os.environ.get("wandb_key")

wandb.login(key=wandb_key)
wandb.init(project="24f2001460-t22026", name="qwen2.5-7b-qlora-mc")

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
MAX_LEN = 512
OPTIONS = ["A", "B", "C", "D", "E"]

train_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)

OPTION_TOKEN_IDS = {o: tokenizer.encode(o, add_special_tokens=False)[0] for o in OPTIONS}

def build_prompt(row):
    return (
        f"Question: {row['prompt']}\n"
        f"A. {row['A']}\n"
        f"B. {row['B']}\n"
        f"C. {row['C']}\n"
        f"D. {row['D']}\n"
        f"E. {row['E']}\n"
        f"Answer:"
    )

def preprocess_train(examples):
    input_ids_list, labels_list, attn_list = [], [], []
    for i in range(len(examples["prompt"])):
        row = {k: examples[k][i] for k in examples}
        prompt = build_prompt(row)
        full_text = prompt + " " + row["answer"]

        prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
        full_ids = tokenizer(full_text, add_special_tokens=False)["input_ids"]
        full_ids = full_ids[:MAX_LEN]

        labels = [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]
        labels = labels[:MAX_LEN]

        attn = [1] * len(full_ids)

        input_ids_list.append(full_ids)
        labels_list.append(labels)
        attn_list.append(attn)

    return {"input_ids": input_ids_list, "labels": labels_list, "attention_mask": attn_list}

train_dataset = Dataset.from_pandas(train_df)
train_dataset = train_dataset.map(
    preprocess_train, batched=True, remove_columns=train_df.columns.tolist()
)

split = train_dataset.train_test_split(test_size=0.1, seed=42)
train_ds, val_ds = split["train"], split["test"]

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer, padding=True, label_pad_token_id=-100
)

args = TrainingArguments(
    output_dir="/kaggle/working/mcq_qwen_output",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-4,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_ratio=0.05,
    bf16=True,
    optim="paged_adamw_8bit",
    report_to="wandb",
    logging_steps=10,
    save_total_limit=1,
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=data_collator,
)

trainer.train()

model.eval()
option_ids_tensor = torch.tensor([OPTION_TOKEN_IDS[o] for o in OPTIONS])

def predict_top3(row):
    prompt = build_prompt(row) + " "
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        logits = model(**inputs).logits[0, -1]
    option_logits = logits[option_ids_tensor.to(model.device)]
    probs = torch.softmax(option_logits, dim=0).float().cpu().numpy()
    top3_idx = np.argsort(-probs)[:3]
    return " ".join(OPTIONS[i] for i in top3_idx)

test_ids = test_df["id"].tolist()
predictions = [predict_top3(row) for _, row in test_df.iterrows()]

submission = pd.DataFrame({"id": test_ids, "Prediction": predictions})
submission.to_csv("submission.csv", index=False)

wandb.finish()

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin
wandb: Tracking run with wandb version 0.25.0
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260715_145033-vmzhie3a
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run qwen2.5-7b-qlora-mc
wandb: ⭐️ View project at https://wandb.ai/24f2001460-iitm/24f2001460-t22026
wandb: 🚀 View run at https://wandb.ai/24f2001460-iitm/24f2001460-t22026/runs/vmzhie3a


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: The AccumulateGrad node's stream does not match the stream of the node th

Epoch,Training Loss,Validation Loss
1,0.087066,nan
2,0.000002,nan
3,0.000003,nan


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
wandb: updating ru

In [4]:
# from kaggle_secrets import UserSecretsClient
# user_secrets = UserSecretsClient()
# secret_value_0 = user_secrets.get_secret("new_token")


In [5]:
# !git clone https://{secret_value_0}@github.com/24f2001460/dl-24f2001460-notebook-t22026.git /kaggle/working/repo1
# %cd /kaggle/working/repo1

In [6]:
# # ── 3. Run inference ───────────────────────────────────────────────────────
# # Adjust --model_path to wherever Kaggle mounted your saved tfidf.pkl dataset
# !python -m src.tfidf_inference\
#     --test_path  /kaggle/input/competitions/smart-mcq-solver-challenge/test.csv \
#     --model_path  /kaggle/input/notebooks/jigyasadoharey/tfidf-model/tfidf.pkl \
#     --output_path /kaggle/working/submission.csv

In [7]:
# !python -m src.scratch_inference\
#     --test_path  /kaggle/input/competitions/smart-mcq-solver-challenge/test.csv \
#     --model_path  /kaggle/input/notebooks/jigyasadoharey/scratch-model/model.pt\
#     --vocab_path /kaggle/input/notebooks/jigyasadoharey/scratch-model/word2idx.pkl\
#     --output_path /kaggle/working/submission.csv